# Step 1: Create a clean, usable dataset
### - Load in the Excel file
### - Convert to Pandas (Language of Choice)
### - Combine all three tabs into one source of data (adding indicators of which year they come from)
### - Standardize columns to create an organized dataset
    - Convert Year to the actual year that corresponds with the date (Ex: 1 -> 2021)
    - Convert Day of Week to a string
    - Convert all ticket volumes to integers with clear labels
    - Convert all ticket revenue to floats with 2 decimals to reflect currency
### - Add additional columns
    - Extract Month from Game Date
    - Classify each game as Weekend or Weekday (Friday, Saturday, Sunday = Weekend; Monday–Thursday = Weekday)
    - Classify each game as a Day Game or Night Game (before 5:00 PM local = Day Game, 5:00 PM or later = Night Game)
    - Total Tickets (#) and Total Revenue ($) — summed across all four ticket segments
    - Average Ticket Price ($) — blended, Total Revenue ÷ Total Tickets
    - Average price per ticket segment (Full Season, Partial Season, Group, Single Game) — each segment's Revenue ÷ Tickets
    - Actual Attendance (#) — Total Tickets × (1 − No-Show Rate)
### - Identify and Correct Data Quality Issues
    - Flag statistical outliers, excluding each season's first calendar month
    - Corrected the one confirmed data entry error found (Group Tickets, Row 43)
### - Save the Clean Data as CSV File for Analysis


### - Load in the Excel file

In [3]:
import pandas as pd

xls = pd.ExcelFile('../../provided_materials/S&A Case Study Data_August 2026 (1).xlsx')
print(xls.sheet_names)

['Data Definitions', 'Year 1 Season Data', 'Year 2 Season Data', 'Year 3 Season Data']


### - Convert to Pandas (Language of Choice)

In [4]:
year_1 = pd.read_excel(xls, sheet_name='Year 1 Season Data')
year_2 = pd.read_excel(xls, sheet_name='Year 2 Season Data')
year_3 = pd.read_excel(xls, sheet_name='Year 3 Season Data')

print(year_1.shape, year_2.shape, year_3.shape)

(81, 13) (81, 13) (81, 13)


### - Combine all three tabs into one source of data (adding indicators of which year they come from)

In [5]:
year_1.insert(0, 'Year', 1)
year_2.insert(0, 'Year', 2)
year_3.insert(0, 'Year', 3)

full = pd.concat([year_1, year_2, year_3], ignore_index=True)
print(f"Combined dataset: {full.shape[0]} rows, {full.shape[1]} columns")
full

Combined dataset: 243 rows, 14 columns


,Year,Home Game #,Game Date,Day of Week,Game Time,Full Season Plans (Tickets),Full Season Plans (Revenue),Partial Season Plans (Tickets),Partial Season Plans (Revenue),Groups (Tickets),Groups (Revenue),Single Game (Tickets),Single Game (Revenue),No-Show Rate
0,1,1,2021-04-01,1900-01-05,13:10:00,9361,417794.037885,5132,191572.895,5279,247218.9750,16916,1.013907e+06,0.141965
1,1,2,2021-04-02,1900-01-06,19:10:00,9374,415278.623385,4201,114894.650,1922,49845.3900,9023,2.728003e+05,0.175194
2,1,3,2021-04-03,1900-01-07,18:10:00,9398,414149.537385,4998,143615.635,4120,125700.1900,10411,3.737338e+05,0.164260
3,1,4,2021-04-04,1900-01-01,13:10:00,9374,415431.578385,4729,161003.200,2309,69639.6950,12149,4.111550e+05,0.160931
4,1,5,2021-04-09,1900-01-06,19:10:00,9376,416211.648885,3446,111596.465,6806,206055.8875,10282,4.080094e+05,0.157028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,3,77,2023-09-19,1900-01-03,19:10:00,8659,407502.000000,3185,109037.000,2373,54404.0000,6633,1.008410e+05,0.182429
239,3,78,2023-09-20,1900-01-04,13:10:00,8499,424941.000000,2409,75514.000,6658,170992.0000,5959,1.517900e+05,0.169830
240,3,79,2023-09-22,1900-01-06,19:10:00,8841,446221.000000,4675,192562.000,7101,273603.0000,9886,4.632760e+05,0.157089
241,3,80,2023-09-23,1900-01-07,18:10:00,8811,416188.000000,4820,200808.000,7123,279110.0000,10680,4.543160e+05,0.154406


### - Standardize columns to create an organized dataset
    - Convert Year to the actual year that corresponds with the date (Ex: 1 -> 2021)
    - Convert Day of Week to a string
    - Convert all ticket volumes to integers with clear labels
    - Convert all ticket revenue to floats with 2 decimals to reflect currency

In [6]:
# ============================================================
# 1. Convert Year to the actual calendar year (e.g., 1 -> 2021)
# ============================================================
full['Year'] = full['Game Date'].dt.year

# ============================================================
# 2. Convert Day of Week to a proper string
#    (original column was corrupted — Excel date serials, not real day names)
# ============================================================
full['Day of Week'] = full['Game Date'].dt.day_name()

# ============================================================
# 3. Convert all ticket volume columns to integers with clear labels (#)
# ============================================================
ticket_cols = {
    'Full Season Plans (Tickets)': 'Full Season Tickets (#)',
    'Partial Season Plans (Tickets)': 'Partial Season Tickets (#)',
    'Groups (Tickets)': 'Group Tickets (#)',
    'Single Game (Tickets)': 'Single Game Tickets (#)'
}
full = full.rename(columns=ticket_cols)
for col in ticket_cols.values():
    full[col] = full[col].astype(int)

# ============================================================
# 4. Convert all ticket revenue columns to floats with 2 decimals ($)
# ============================================================
revenue_cols = {
    'Full Season Plans (Revenue)': 'Full Season Revenue ($)',
    'Partial Season Plans (Revenue)': 'Partial Season Revenue ($)',
    'Groups (Revenue)': 'Group Revenue ($)',
    'Single Game (Revenue)': 'Single Game Revenue ($)'
}
full = full.rename(columns=revenue_cols)
for col in revenue_cols.values():
    full[col] = full[col].astype(float).round(2)

print(f"Standardized dataset: {full.shape[0]} rows, {full.shape[1]} columns")
full.head()

Standardized dataset: 243 rows, 14 columns


,Year,Home Game #,Game Date,Day of Week,Game Time,Full Season Tickets (#),Full Season Revenue ($),Partial Season Tickets (#),Partial Season Revenue ($),Group Tickets (#),Group Revenue ($),Single Game Tickets (#),Single Game Revenue ($),No-Show Rate
0,2021,1,2021-04-01,Thursday,13:10:00,9361,417794.04,5132,191572.90,5279,247218.97,16916,1013906.60,0.141965
1,2021,2,2021-04-02,Friday,19:10:00,9374,415278.62,4201,114894.65,1922,49845.39,9023,272800.32,0.175194
2,2021,3,2021-04-03,Saturday,18:10:00,9398,414149.54,4998,143615.64,4120,125700.19,10411,373733.78,0.164260
3,2021,4,2021-04-04,Sunday,13:10:00,9374,415431.58,4729,161003.20,2309,69639.69,12149,411154.96,0.160931
4,2021,5,2021-04-09,Friday,19:10:00,9376,416211.65,3446,111596.46,6806,206055.89,10282,408009.37,0.157028


### - Add additional columns
    - Extract Month from Game Date
    - Classify each game as Weekend or Weekday (Friday, Saturday, Sunday = Weekend; Monday–Thursday = Weekday)
    - Classify each game as a Day Game or Night Game (before 5:00 PM local = Day Game, 5:00 PM or later = Night Game)
    - Total Tickets (#) and Total Revenue ($) — summed across all four ticket segments
    - Average Ticket Price ($) — blended, Total Revenue ÷ Total Tickets
    - Average price per ticket segment (Full Season, Partial Season, Group, Single Game) — each segment's Revenue ÷ Tickets
    - Actual Attendance (#) — Total Tickets × (1 − No-Show Rate)

In [7]:
# ============================================================
# Month
# ============================================================
full['Month'] = full['Game Date'].dt.month_name()

# ============================================================
# Weekend / Weekday
# ============================================================
full['Weekend / Weekday'] = full['Day of Week'].apply(
    lambda d: 'Weekend' if d in ['Friday', 'Saturday', 'Sunday'] else 'Weekday'
)

# ============================================================
# Day Game / Night Game
# ============================================================
full['Day/Night'] = full['Game Time'].apply(
    lambda t: 'Day Game' if t.hour < 17 else 'Night Game'
)

# ============================================================
# Average price per ticket segment
# ============================================================
full['Full Season Avg Price ($)'] = (full['Full Season Revenue ($)'] / full['Full Season Tickets (#)']).round(2)
full['Partial Season Avg Price ($)'] = (full['Partial Season Revenue ($)'] / full['Partial Season Tickets (#)']).round(2)
full['Group Avg Price ($)'] = (full['Group Revenue ($)'] / full['Group Tickets (#)']).round(2)
full['Single Game Avg Price ($)'] = (full['Single Game Revenue ($)'] / full['Single Game Tickets (#)']).round(2)

# ============================================================
# No-Show Rate — round to 4 decimal places
# ============================================================
full['No-Show Rate'] = full['No-Show Rate'].round(4)

# ============================================================
# Totals (rounded to avoid floating-point precision artifacts from summing rounded values)
# ============================================================
full['Total Tickets (#)'] = (full['Full Season Tickets (#)'] + full['Partial Season Tickets (#)']
                              + full['Group Tickets (#)'] + full['Single Game Tickets (#)'])
full['Total Revenue ($)'] = (full['Full Season Revenue ($)'] + full['Partial Season Revenue ($)']
                              + full['Group Revenue ($)'] + full['Single Game Revenue ($)']).round(2)

# ============================================================
# Actual Attendance (accounting for no-shows) — whole people, so integer
# ============================================================
full['Actual Attendance (#)'] = (full['Total Tickets (#)'] * (1 - full['No-Show Rate'])).round(0).astype(int)

# ============================================================
# Average Ticket Price (blended, across all segments)
# ============================================================
full['Average Ticket Price ($)'] = (full['Total Revenue ($)'] / full['Total Tickets (#)']).round(2)

# ============================================================
# Final column order
# ============================================================
final_column_order = [
    'Home Game #', 'Game Date', 'Year', 'Month',
    'Day of Week', 'Weekend / Weekday',
    'Game Time', 'Day/Night',
    'Full Season Tickets (#)', 'Full Season Revenue ($)', 'Full Season Avg Price ($)',
    'Partial Season Tickets (#)', 'Partial Season Revenue ($)', 'Partial Season Avg Price ($)',
    'Group Tickets (#)', 'Group Revenue ($)', 'Group Avg Price ($)',
    'Single Game Tickets (#)', 'Single Game Revenue ($)', 'Single Game Avg Price ($)',
    'No-Show Rate',
    'Total Tickets (#)', 'Actual Attendance (#)',
    'Total Revenue ($)', 'Average Ticket Price ($)'
]
full = full[final_column_order]

print(f"Final dataset: {full.shape[0]} rows, {full.shape[1]} columns")
full

Final dataset: 243 rows, 25 columns


,Home Game #,Game Date,Year,Month,Day of Week,Weekend / Weekday,Game Time,Day/Night,Full Season Tickets (#),Full Season Revenue ($),...,Group Revenue ($),Group Avg Price ($),Single Game Tickets (#),Single Game Revenue ($),Single Game Avg Price ($),No-Show Rate,Total Tickets (#),Actual Attendance (#),Total Revenue ($),Average Ticket Price ($)
0,1,2021-04-01,2021,April,Thursday,Weekday,13:10:00,Day Game,9361,417794.04,...,247218.97,46.83,16916,1013906.60,59.94,0.1420,36688,31478,1870492.51,50.98
1,2,2021-04-02,2021,April,Friday,Weekend,19:10:00,Night Game,9374,415278.62,...,49845.39,25.93,9023,272800.32,30.23,0.1752,24520,20224,852818.98,34.78
2,3,2021-04-03,2021,April,Saturday,Weekend,18:10:00,Night Game,9398,414149.54,...,125700.19,30.51,10411,373733.78,35.90,0.1643,28927,24174,1057199.15,36.55
3,4,2021-04-04,2021,April,Sunday,Weekend,13:10:00,Day Game,9374,415431.58,...,69639.69,30.16,12149,411154.96,33.84,0.1609,28561,23966,1057229.43,37.02
4,5,2021-04-09,2021,April,Friday,Weekend,19:10:00,Night Game,9376,416211.65,...,206055.89,30.28,10282,408009.37,39.68,0.1570,29910,25214,1141873.37,38.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,77,2023-09-19,2023,September,Tuesday,Weekday,19:10:00,Night Game,8659,407502.00,...,54404.00,22.93,6633,100841.00,15.20,0.1824,20850,17047,671784.00,32.22
239,78,2023-09-20,2023,September,Wednesday,Weekday,13:10:00,Day Game,8499,424941.00,...,170992.00,25.68,5959,151790.00,25.47,0.1698,23525,19530,823237.00,34.99
240,79,2023-09-22,2023,September,Friday,Weekend,19:10:00,Night Game,8841,446221.00,...,273603.00,38.53,9886,463276.00,46.86,0.1571,30503,25711,1375662.00,45.10
241,80,2023-09-23,2023,September,Saturday,Weekend,18:10:00,Night Game,8811,416188.00,...,279110.00,39.18,10680,454316.00,42.54,0.1544,31434,26581,1350422.00,42.96


### - Identify and Correct Data Quality Issues
    - Flag statistical outliers using IQR, excluding each season's first calendar month (Opening Day demand is a real, expected spike, not an error)
    - Correct the one confirmed data entry error found (Group Tickets, Row 43)

In [8]:
# Identify the first calendar month of each season (excludes Opening Day/opening-week demand
# spikes from being flagged as data errors, since that's a real, expected seasonal pattern)
full['Is First Month'] = full.apply(
    lambda row: row['Game Date'].month == full[full['Year'] == row['Year']]['Game Date'].dt.month.min(),
    axis=1
)

# Columns to check for outliers
cols_to_check = [
    'Full Season Tickets (#)', 'Partial Season Tickets (#)', 'Group Tickets (#)', 'Single Game Tickets (#)',
    'Full Season Revenue ($)', 'Partial Season Revenue ($)', 'Group Revenue ($)', 'Single Game Revenue ($)',
    'Full Season Avg Price ($)', 'Partial Season Avg Price ($)', 'Group Avg Price ($)', 'Single Game Avg Price ($)'
]

# Run IQR-based outlier detection only on rows OUTSIDE the first month
non_first_month = full[~full['Is First Month']]

outlier_flags = pd.DataFrame(index=full.index)
for col in cols_to_check:
    q1, q3 = non_first_month[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
    outlier_flags[col] = (~full['Is First Month']) & ((full[col] < lower) | (full[col] > upper))

outlier_rows = full[outlier_flags.any(axis=1)]
print(f"Found {len(outlier_rows)} outlier row(s) outside the first month:")
print(outlier_rows[['Year', 'Home Game #', 'Game Date']].to_string())

# Known correction: Row 43 (2021, Game 44) — Group Tickets appears to be a data entry error
# (174 tickets implies an implausible $1,396/ticket average against otherwise-normal revenue).
# Replacing with that season's average Group Tickets count as a reasonable estimate.
year_avg_group_tickets = full[(full['Year'] == 2021) & (~full['Is First Month'])]['Group Tickets (#)'].mean()
full.loc[43, 'Group Tickets (#)'] = round(year_avg_group_tickets)
full.loc[43, 'Group Avg Price ($)'] = round(full.loc[43, 'Group Revenue ($)'] / full.loc[43, 'Group Tickets (#)'], 2)

print(f"\nCorrected Row 43 Group Tickets to {full.loc[43, 'Group Tickets (#)']:.0f} (season average)")

Found 1 outlier row(s) outside the first month:
    Year  Home Game #  Game Date
43  2021           44 2021-07-03

Corrected Row 43 Group Tickets to 5626 (season average)


### - Save the Clean Data as CSV File for Analysis

In [9]:
# ============================================================
# Save cleaned dataset for use in analysis notebook
# ============================================================
full.to_csv('clean_data.csv', index=False)
print(f"Saved clean_data.csv — {full.shape[0]} rows, {full.shape[1]} columns")

Saved clean_data.csv — 243 rows, 26 columns
